# Failure Structure
Configuration: `stages/01_failure_structure/config/runtime.json`.


### Initialization

Run the following setup cells from the Decoders working directory in the qec environment. They load this notebook's configuration, paths and dependencies; subsequent experiments reuse these definitions and should be read in execution order.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
failure_structure_experiment_dir = ROOT / 'experiment/research/trapping_sets/stages/01_failure_structure'
sys.path[:0] = [str(ROOT / 'src'), str(ROOT), str(ROOT / 'experiment')]

from experiment.utils.data_paths import ExperimentPaths

failure_structure_data_paths = ExperimentPaths(ROOT, failure_structure_experiment_dir / 'config/paths.json')

import json

failure_structure_config = json.loads((failure_structure_experiment_dir / 'config/runtime.json').read_text())
# Expected configuration values:
# print_level: integer; 1 = silent, 2 = key information, 3 = debug.

from tqdm.auto import tqdm
from experiment.utils.trapping_sets import save_plotly

failure_structure_plots = failure_structure_data_paths.plots()
failure_structure_plots.mkdir(parents=True, exist_ok=True)


In [ ]:
from experiment.utils import misc

def failure_structure_aggregate_path(path):
    return misc.aggregate_path(path, data_paths=failure_structure_data_paths)


# Graph Structure

In [ ]:
import multiprocessing
import sys
from pathlib import Path
from collections import defaultdict, Counter
from itertools import combinations
from typing import Literal
import numpy as np
import pandas as pd
import sinter
import stim
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from utils.graph_vis import visualize_det_on_graph_plotly
from codes.ca_code import CA_5_2_3, CA_9_2_5
from src.noises_ca import CircuitBuilder, ErrorRate

%load_ext autoreload
%autoreload 2


In [ ]:
failure_structure_ca_codes = {'CA_5_2_3': CA_5_2_3(), 'CA_9_2_5': CA_9_2_5()}
failure_structure_p_values = np.geomspace(1e-08, 0.1, 12)
failure_structure_max_shots = 1000000000
max_errors = 2000


In [ ]:
from experiment.utils import dem_graph

extract_dem_info = dem_graph.extract_dem_info
extract_coord_from_graph = dem_graph.extract_coord_from_graph

def failure_structure_build_tanner_graph_from_dem(dem):
    return dem_graph.build_tanner_graph_from_dem(dem, extract_dem_info=extract_dem_info)

from utils.graph_struct import induced_subgraph

def induced_trapping_subgraph(
    G: nx.Graph,
    active_dets: set[int],
) -> tuple[nx.Graph, set[int], set[int]]:
    """
    Returns the trapping subgraph of G induced by the active detectors.

    The trapping subgraph comes from the error nodes (varible nodes) induced by active detectors
    """

    SG = induced_subgraph(G, active_dets)
    error_nodes = set([n for n, d in SG.nodes(data=True) if d.get('kind') == 'error'])
    affected_edges = [(u, v) for u, v in SG.edges if u in error_nodes or v in error_nodes]
    trapping_SG = G.edge_subgraph(affected_edges).copy()
    check_nodes = set(trapping_SG.nodes()) - error_nodes

    return (trapping_SG, error_nodes, check_nodes)

event_to_active_ids = dem_graph.event_to_active_ids

def graph_det_features(G: nx.Graph, det_events: str):
    """
    Extracts Tanner trapping-subgraph features from the active detector set.

    Parameters
    ----------
    G: nx.Graph
        the tanner graph built from the DEM.
    det_events: str
        string of '0' and '1' representing detector events.

    Returns
    ----------
    features: dict
        dictionary of extracted features, including:
        * WL_hash                          (str): Weisfeiler-Lehman graph hash of the induced subgraph, considering 'active' as edge attribute.
        * active_detectors                 (int): number of active detectors
        * induced_nodes                    (int): number of nodes in the subgraph induced by active detectors
        * induced_edges                    (int): number of edges in the subgraph induced by active detectors
        * components                       (int): number of connected components in the induced subgraph
        * largest_component_size           (int): size of the largest connected component in the induced subgraph
        * cycle_rank                       (int): the number of independent cycles in the subgraph
        * inverse_girth                  (float): the inverse of the length of the shortest cycle in the subgraph (0 if no cycle)
        * density                        (float): density of the induced subgraph, defined as 2*|E|/(|V|*(|V|-1)) 
        * mean_base_degree_on_active     (float): mean degree of active detectors in the original graph G
        * trapping_param                 (tuple): (a, b) where a = number of error nodes in the trapping subgraph, b = number of odd-degree check nodes in the trapping subgraph
        * error_node_parity              (tuple): (n_even, n_odd) where n_even/odd is the number of error nodes in the trapping subgraph with even/odd degree
    """

    def is_distance_two(graph, u, v):
        """Checks if nodes u and v are at distance 2 in the graph (i.e., they are not directly connected but share a common neighbor)."""
        if v in graph[u]:
            return False

        for w in tqdm(graph[u], desc='Trapping-set analysis', leave=True, disable=failure_structure_config['print_level'] == 1):
            if w in graph[v]:
                return True

        return False

    active_dets = set(event_to_active_ids(det_events))
    n_active = len(active_dets)
    if n_active == 0:
        return {
            'WL_hash': '',
            'detector_events': det_events,
            'active_detectors': 0,
            'induced_nodes': 0,
            'induced_edges': 0,
            'components': 0,
            'largest_component_size': 0,
            'cycle_rank': 0,
            'inverse_girth': 0.0,
            'density': 0.0,
            'mean_base_degree_on_active': 0.0,
            'trapping_param': (),
            'error_node_parity': (0, 0),
        }

    SG = induced_subgraph(G, active_dets)
    comps = list(nx.connected_components(SG))
    largest_comp_size = max((len(c) for c in comps)) if comps else 0
    n_comp = len(comps)
    induced_edges = SG.number_of_edges()
    induced_nodes = SG.number_of_nodes()
    cycle_rank = induced_edges - induced_nodes + n_comp
    girth = nx.girth(SG)
    ne, nv = (sum((is_distance_two(G, u, v) for u, v in combinations(active_dets, 2))), n_active)
    density = ne / (nv * (nv - 1)) if nv > 1 else 0.0
    base_degrees = [G.degree(d) for d in active_dets]
    trapping_SG, error_nodes, check_nodes = induced_trapping_subgraph(G, active_dets)
    trapping_a = len(error_nodes)
    trapping_b = sum((1 for n in check_nodes if trapping_SG.degree(n) % 2 == 1))

    return {
        'WL_hash': nx.weisfeiler_lehman_graph_hash(SG, node_attr='active'),
        'detector_events': det_events,
        'active_detectors': n_active,
        'induced_nodes': induced_nodes,
        'induced_edges': induced_edges,
        'components': n_comp,
        'largest_component_size': largest_comp_size,
        'cycle_rank': cycle_rank,
        'inverse_girth': 1 / girth if np.isfinite(girth) else 0.0,
        'density': density,
        'mean_base_degree_on_active': float(np.mean(base_degrees)) if len(base_degrees) else 0.0,
        'trapping_param': (trapping_a, trapping_b),
        'error_node_parity': (sum((1 for n in error_nodes if trapping_SG.degree(n) % 2 == 0)), sum((1 for n in error_nodes if trapping_SG.degree(n) % 2 == 1))),
    }

def graph_features_no_det_event(G: nx.Graph):

    def is_distance_two(graph, u, v):
        """Checks if nodes u and v are at distance 2 in the graph (i.e., they are not directly connected but share a common neighbor)."""
        if v in graph[u]:
            return False

        for w in tqdm(graph[u], desc='Trapping-set analysis', leave=True, disable=failure_structure_config['print_level'] == 1):
            if w in graph[v]:
                return True

        return False

    comps = list(nx.connected_components(G))
    largest_comp_size = max((len(c) for c in comps)) if comps else 0
    n_comp = len(comps)
    edges = G.number_of_edges()
    nodes = G.number_of_nodes()
    cycle_rank = edges - nodes + n_comp
    girth = nx.girth(G)
    ne = sum((is_distance_two(G, u, v) for u, v in combinations(G.nodes, 2)))
    density = ne / (nodes * (nodes - 1)) if nodes > 1 else 0.0
    variable_nodes = set((n for n, d in G.nodes(data=True) if d.get('kind') == 'error'))
    check_nodes = set((n for n, d in G.nodes(data=True) if d.get('kind') == 'detector'))
    trapping_a = len(variable_nodes)
    trapping_b = sum((1 for n in check_nodes if G.degree(n) % 2 == 1))

    return {
        'nodes': nodes,
        'edges': edges,
        'components': n_comp,
        'largest_component_size': largest_comp_size,
        'cycle_rank': cycle_rank,
        'inverse_girth': 1 / girth if np.isfinite(girth) else 0.0,
        'density': density,
        'trapping_param': (trapping_a, trapping_b),
        'error_node_parity': (sum((1 for n in variable_nodes if G.degree(n) % 2 == 0)), sum((1 for n in variable_nodes if G.degree(n) % 2 == 1))),
    }


## BPOSD Failure-Region Graph Analysis

- `bfts_cases`: BPOSD fails but Tesseract succeeds (target failure region)
- `bs_cases`: both decoders succeed (control)

### Compute empirical graph features

Load outcome-labeled case populations and extract their graph descriptors. The comparison needs success and reverse-disagreement controls as well as the target failure group.


In [ ]:
failure_structure_data_dir = failure_structure_data_paths.output('circuit_level')
failure_structure_cases_dir = failure_structure_data_paths.input('failure_cases')
assert failure_structure_cases_dir.is_dir(), failure_structure_cases_dir


In [ ]:
failure_structure_labels = {
    '00': 'bf',
    '11': 'bs',
    '01': 'bfts',
    '10': 'bstf',
}


In [ ]:
failure_structure_bfts_cases_path = failure_structure_data_paths.input('ca_bfts_cases_csv')
failure_structure_bstf_cases_path = failure_structure_data_paths.input('ca_bstf_cases_csv')
failure_structure_bs_cases_path = failure_structure_data_paths.input('ca_bs_cases_csv')
failure_structure_bf_cases_path = failure_structure_data_paths.input('ca_bf_cases_csv')
failure_structure_types = {
    'code': str,
    'p': float,
    'detector_events': str,
    'label': str,
}

failure_structure_bfts_cases = pd.read_csv(failure_structure_bfts_cases_path, dtype=failure_structure_types)
failure_structure_bstf_cases = pd.read_csv(failure_structure_bstf_cases_path, dtype=failure_structure_types)
failure_structure_bs_cases = pd.read_csv(failure_structure_bs_cases_path, dtype=failure_structure_types)
failure_structure_bf_cases = pd.read_csv(failure_structure_bf_cases_path, dtype=failure_structure_types)

failure_structure_bfts_cases['label'] = '01'
failure_structure_bstf_cases['label'] = '10'
failure_structure_bs_cases['label'] = '11'
failure_structure_bf_cases['label'] = '00'
failure_structure_bfts_cases.drop_duplicates(subset=['code', 'p', 'detector_events', 'label'], inplace=True)
failure_structure_bstf_cases.drop_duplicates(subset=['code', 'p', 'detector_events', 'label'], inplace=True)
failure_structure_bs_cases.drop_duplicates(subset=['code', 'p', 'detector_events', 'label'], inplace=True)
failure_structure_bf_cases.drop_duplicates(subset=['code', 'p', 'detector_events', 'label'], inplace=True)


In [ ]:
analysis_rows = []
for (code_name, failure_structure_p), grp_fail in tqdm(failure_structure_bfts_cases.groupby(['code', 'p']), desc='Trapping-set analysis', leave=True, disable=failure_structure_config['print_level'] == 1):
    if failure_structure_config['print_level'] >= 2:
        print(f'Analyzing failure cases for code={code_name}, p={failure_structure_p:.6f} with {len(grp_fail)} samples')
    failure_structure_code = failure_structure_ca_codes[code_name]
    failure_structure_builder = CircuitBuilder(failure_structure_code)
    failure_structure_circ = failure_structure_builder.get_circuit_level_circuit(error_rate=ErrorRate(p=failure_structure_p), d_rounds=3)
    failure_structure_dem = failure_structure_circ.detector_error_model()
    failure_structure_g, failure_structure_ = failure_structure_build_tanner_graph_from_dem(failure_structure_dem)
    ctrl_df = failure_structure_bs_cases[(failure_structure_bs_cases['code'] == code_name) & (failure_structure_bs_cases['p'] == failure_structure_p)][['code', 'p', 'detector_events', 'label']].copy()
    fail_df = grp_fail[['code', 'p', 'detector_events', 'label']].copy()
    merged = pd.concat([fail_df, ctrl_df], ignore_index=True)

    for failure_structure_, failure_structure_r in tqdm(merged.iterrows(), desc='Trapping-set analysis', leave=False, disable=failure_structure_config['print_level'] == 1):
        feats = graph_det_features(failure_structure_g, failure_structure_r['detector_events'])
        feats.update({
            'code': code_name,
            'p': float(failure_structure_p),
            'label': failure_structure_r['label'],
        })
        analysis_rows.append(feats)

analysis_df = pd.DataFrame(analysis_rows)
analysis_df.head()


In [ ]:
display(analysis_df['cycle_rank'].value_counts())
df = analysis_df[['code', 'p', 'cycle_rank', 'label']].copy()
failure_structure_fig = sns.relplot(
    data=df,
    x='p',
    y='cycle_rank',
    hue='label',
    col='code',
    kind='line',
    marker='o',
    facet_kws={'sharey': False, 'sharex': True},
)
failure_structure_fig.set_titles('code = {col_name}')
failure_structure_fig.set_axis_labels('p', 'cycle_rank')

plt.savefig(failure_structure_data_paths.output('cell_0011_1_png'), bbox_inches='tight')
plt.show()


In [ ]:
summary_cmp = analysis_df.groupby(['code', 'p', 'label']).mean(numeric_only=True).reset_index()
pivot_fail = summary_cmp[summary_cmp['label'] == '01'].set_index(['code', 'p'])
pivot_ctrl = summary_cmp[summary_cmp['label'] == '11'].set_index(['code', 'p'])
common_idx = pivot_fail.index.intersection(pivot_ctrl.index)
cols = summary_cmp.columns.difference(['code', 'p', 'label'])
diff_df = (pivot_fail.loc[common_idx, cols] - pivot_ctrl.loc[common_idx, cols]).reset_index()
diff_df.head()


In [ ]:
plot_feats = ['components', 'largest_component_size', 'cycle_rank', 'density', 'inverse_girth', 'induced_nodes', 'induced_edges', 'mean_base_degree_on_active']
failure_structure_fig, axes = plt.subplots(len(plot_feats), 1, figsize=(10, 3 * len(plot_feats)), squeeze=False)
for failure_structure_i, f in tqdm(enumerate(plot_feats), desc='Trapping-set analysis', leave=True, disable=failure_structure_config['print_level'] == 1):
    failure_structure_ax = axes[failure_structure_i, 0]
    for code_name, g in tqdm(diff_df.groupby('code'), desc='Trapping-set analysis', leave=False, disable=failure_structure_config['print_level'] == 1):
        gg = g.sort_values('p')
        failure_structure_ax.plot(gg['p'], gg[f], marker='o', label=code_name if failure_structure_i == 0 else None)

    failure_structure_ax.axhline(0.0, color='black', lw=1)
    failure_structure_ax.set_xscale('log')
    failure_structure_ax.set_ylabel(f + ' (fail-control)')
    failure_structure_ax.grid(True, which='both', ls='--', alpha=0.35)

axes[-1, 0].set_xlabel('physical error rate p')
if len(diff_df) > 0:
    axes[0, 0].legend()
plt.tight_layout()

plt.savefig(failure_structure_data_paths.output('cell_0013_1_png'), bbox_inches='tight')
plt.show()


## Correlation of the metrics

1. pairwise correlation
2. PCA on the metrics

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

feats = ['components', 'largest_component_size', 'cycle_rank', 'density', 'inverse_girth', 'induced_nodes', 'induced_edges', 'mean_base_degree_on_active']


In [ ]:
set1 = set(map(tuple, failure_structure_bstf_cases[['code', 'p']].drop_duplicates().values))
set2 = set(map(tuple, failure_structure_bfts_cases[['code', 'p']].drop_duplicates().values))
set3 = set(map(tuple, failure_structure_bs_cases[['code', 'p']].drop_duplicates().values))
set4 = set(map(tuple, failure_structure_bf_cases[['code', 'p']].drop_duplicates().values))
common_set = set1 & set2 & set3 & set4
common = pd.DataFrame(list(common_set), columns=['code', 'p'])
common.sort_values(['code', 'p'], inplace=True)
common


In [ ]:
analysis_rows = []
for failure_structure_, (code_name, failure_structure_p) in tqdm(common.iterrows(), desc='Trapping-set analysis', leave=True, disable=failure_structure_config['print_level'] == 1):
    if failure_structure_config['print_level'] >= 2:
        print(f'Analyzing cases for code={code_name}, p={failure_structure_p:.6f}')
    failure_structure_code = failure_structure_ca_codes[code_name]
    failure_structure_builder = CircuitBuilder(failure_structure_code)
    failure_structure_circ = failure_structure_builder.get_circuit_level_circuit(error_rate=ErrorRate(p=failure_structure_p), d_rounds=3)
    failure_structure_dem = failure_structure_circ.detector_error_model()
    failure_structure_g, failure_structure_ = failure_structure_build_tanner_graph_from_dem(failure_structure_dem)
    failure_structure_bs_df = failure_structure_bs_cases[(failure_structure_bs_cases['code'] == code_name) & (failure_structure_bs_cases['p'] == failure_structure_p)][['code', 'p', 'detector_events', 'label']].copy()
    failure_structure_bf_df = failure_structure_bf_cases[(failure_structure_bf_cases['code'] == code_name) & (failure_structure_bf_cases['p'] == failure_structure_p)][['code', 'p', 'detector_events', 'label']].copy()
    failure_structure_bstf_df = failure_structure_bstf_cases[(failure_structure_bstf_cases['code'] == code_name) & (failure_structure_bstf_cases['p'] == failure_structure_p)][['code', 'p', 'detector_events', 'label']].copy()
    failure_structure_bfts_df = failure_structure_bfts_cases[(failure_structure_bfts_cases['code'] == code_name) & (failure_structure_bfts_cases['p'] == failure_structure_p)][['code', 'p', 'detector_events', 'label']].copy()
    merged = pd.concat([failure_structure_bs_df, failure_structure_bf_df, failure_structure_bstf_df, failure_structure_bfts_df], ignore_index=True)
    merged = merged[merged['detector_events'].str.count('1') > 0]

    for failure_structure_, failure_structure_r in tqdm(merged.iterrows(), desc='Trapping-set analysis', leave=False, disable=failure_structure_config['print_level'] == 1):
        gd_feats = graph_det_features(failure_structure_g, failure_structure_r['detector_events'])
        gd_feats.update({
            'code': code_name,
            'p': float(failure_structure_p),
            'label': failure_structure_r['label'],
        })
        analysis_rows.append(gd_feats)

analysis_df = pd.DataFrame(analysis_rows)
out_dir = failure_structure_data_paths.output('graph_analysis')
out_dir.mkdir(parents=True, exist_ok=True)

analysis_df.to_csv(failure_structure_data_paths.output('event_graph_features_csv'), index=False)
if failure_structure_config['print_level'] >= 2:
    print('saved:', failure_structure_data_paths.output('event_graph_features_csv'))
analysis_df.tail()


In [ ]:
out_dir = failure_structure_data_paths.output('graph_analysis')
analysis_df = pd.read_csv(failure_structure_data_paths.output('event_graph_features_csv'), dtype={'code': str, 'p': float, 'label': str})

grouped_df = analysis_df.groupby(['code', 'p'])
corr_matrices = {}
for (failure_structure_code, failure_structure_p), df in tqdm(grouped_df, desc='Trapping-set analysis', leave=True, disable=failure_structure_config['print_level'] == 1):
    corr = df[feats].corr()
    corr_matrices[failure_structure_code, failure_structure_p] = corr


In [ ]:
cols = 3
rows = int(np.ceil(len(corr_matrices) / cols))
failure_structure_fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4 * rows))
axes = axes.flatten()

for failure_structure_i, key in tqdm(enumerate(corr_matrices.keys()), desc='Trapping-set analysis', leave=True, disable=failure_structure_config['print_level'] == 1):
    failure_structure_code, failure_structure_p = key
    failure_structure_ax = axes[failure_structure_i]
    sns.heatmap(
        corr_matrices[key],
        ax=failure_structure_ax,
        vmin=-1,
        vmax=1,
        cmap='coolwarm',
        square=True,
        cbar=False,
    )
    failure_structure_ax.set_title(f'code={failure_structure_code}, p={failure_structure_p}')

for j in tqdm(range(failure_structure_i + 1, len(axes)), desc='Trapping-set analysis', leave=True, disable=failure_structure_config['print_level'] == 1):
    failure_structure_fig.delaxes(axes[j])
plt.tight_layout()

plt.savefig(failure_structure_data_paths.output('cell_0019_1_png'), bbox_inches='tight')
plt.show()


### PCA

In [ ]:
failure_structure_fig, axes = plt.subplots(1, 2, figsize=(12, 6))
for failure_structure_code in tqdm(analysis_df['code'].unique(), desc='Trapping-set analysis', leave=True, disable=failure_structure_config['print_level'] == 1):
    sub = analysis_df[analysis_df['code'] == failure_structure_code]
    scaler = StandardScaler()
    scaled_feats = scaler.fit_transform(sub[feats])
    pca = PCA(n_components=2)

    X_pca = pca.fit_transform(sub[feats])
    axes[0].scatter(X_pca[:, 0], X_pca[:, 1], label=failure_structure_code)

axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')
axes[0].legend()
axes[0].set_title('PCA of Graph Features by Code')

for failure_structure_p in tqdm(analysis_df['p'].unique(), desc='Trapping-set analysis', leave=True, disable=failure_structure_config['print_level'] == 1):
    sub = analysis_df[analysis_df['p'] == failure_structure_p]
    scaler = StandardScaler()
    scaled_feats = scaler.fit_transform(sub[feats])
    pca = PCA(n_components=2)

    X_pca = pca.fit_transform(sub[feats])
    axes[1].scatter(X_pca[:, 0], X_pca[:, 1], label=failure_structure_p)

axes[1].set_xlabel('PC1')
axes[1].set_ylabel('PC2')
axes[1].legend()
axes[1].set_title('PCA of Graph Features by p')
plt.tight_layout()

plt.savefig(failure_structure_data_paths.output('cell_0021_1_png'), bbox_inches='tight')
plt.show()


In [ ]:
failure_structure_fig, axes = plt.subplots(1, 2, figsize=(12, 6))
codes = analysis_df['code'].unique()
for failure_structure_i, failure_structure_code in tqdm(enumerate(codes), desc='Trapping-set analysis', leave=True, disable=failure_structure_config['print_level'] == 1):
    sub_code = analysis_df[analysis_df['code'] == failure_structure_code]
    for label in tqdm(sub_code['label'].unique(), desc='Trapping-set analysis', leave=False, disable=failure_structure_config['print_level'] == 1):
        sub = sub_code[sub_code['label'] == label]
        scaler = StandardScaler()
        scaled_feats = scaler.fit_transform(sub[feats])
        pca = PCA(n_components=2)

        X_pca = pca.fit_transform(sub[feats])
        axes[failure_structure_i].scatter(X_pca[:, 0], X_pca[:, 1], label=failure_structure_labels[label])

    axes[failure_structure_i].set_xlabel('PC1')
    axes[failure_structure_i].set_ylabel('PC2')
    axes[failure_structure_i].legend()
    axes[failure_structure_i].set_title(f'PCA of Graph Features for code={failure_structure_code}')

plt.savefig(failure_structure_data_paths.output('cell_0022_1_png'), bbox_inches='tight')
plt.show()


## Graph Isomorphism

### Group structural motifs

Compare graph-isomorphism/WL-hash information before interpreting recurrent motifs. A shared hash is a candidate grouping; it is not by itself an isomorphism proof or a causal trapping-set test.


In [ ]:
out_dir = failure_structure_data_paths.output('graph_analysis')
failure_structure_types = {
    'code': str,
    'p': float,
    'label': str,
    'detector_events': str,
    'WL_hash': str,
}

analysis_df = pd.read_csv(failure_structure_data_paths.output('event_graph_features_csv'), dtype=failure_structure_types)
analysis_df = analysis_df[analysis_df['label'] == '01'].drop('label', axis=1)


In [ ]:
graph_hash_counts = analysis_df['WL_hash'].value_counts()
failure_structure_fig, failure_structure_ax = plt.subplots(1, 3, figsize=(18, 6))
failure_structure_ax[0].bar(range(len(graph_hash_counts)), graph_hash_counts.values)
failure_structure_ax[0].set_ylabel('Count of Graph Hash')
graph_hash_counts_code1 = analysis_df[analysis_df['code'] == 'CA_5_2_3']['WL_hash'].value_counts()

failure_structure_ax[1].bar(range(len(graph_hash_counts_code1)), graph_hash_counts_code1.values)
failure_structure_ax[1].set_ylabel('Count of Graph Hash')
failure_structure_ax[1].set_title('Graph Hash Counts for CA_5_2_3')
graph_hash_counts_code2 = analysis_df[analysis_df['code'] == 'CA_9_2_5']['WL_hash'].value_counts()

failure_structure_ax[2].bar(range(len(graph_hash_counts_code2)), graph_hash_counts_code2.values)
failure_structure_ax[2].set_ylabel('Count of Graph Hash')
failure_structure_ax[2].set_title('Graph Hash Counts for CA_9_2_5')

plt.savefig(failure_structure_data_paths.output('cell_0026_1_png'), bbox_inches='tight')
plt.show()


In [ ]:
(graph_hash_counts_code1, graph_hash_counts_code2)


In [ ]:
code1_analysis = analysis_df[analysis_df['code'] == 'CA_5_2_3'].drop('code', axis=1)
code2_analysis = analysis_df[analysis_df['code'] == 'CA_9_2_5'].drop('code', axis=1)


In [ ]:
column_overview_rows = []
for failure_structure_col in tqdm(code1_analysis.columns, desc='Trapping-set analysis', leave=True, disable=failure_structure_config['print_level'] == 1):
    s = code1_analysis[failure_structure_col]
    column_overview_rows.append({
        'column': failure_structure_col,
        'dtype': str(s.dtype),
        'non_null': int(s.notna().sum()),
        'unique_values': int(s.nunique(dropna=True)),
        'sample_values': ', '.join(map(str, s.dropna().head(5).tolist())),
    })

column_overview_df = pd.DataFrame(column_overview_rows)
column_overview_df


In [ ]:
from utils.graph_vis import visualize_det_on_graph_plotly

failure_structure_row = analysis_df[analysis_df['WL_hash'] == '334da058d72778ee6a6c10df2b74817d'].iloc[0]
failure_structure_p = float(failure_structure_row['p'])
failure_structure_code = failure_structure_ca_codes[failure_structure_row['code']]
failure_structure_builder = CircuitBuilder(failure_structure_code)
failure_structure_circ = failure_structure_builder.get_circuit_level_circuit(error_rate=ErrorRate(p=failure_structure_p), d_rounds=3)
failure_structure_dem = failure_structure_circ.detector_error_model()
failure_structure_g, failure_structure_ = failure_structure_build_tanner_graph_from_dem(failure_structure_dem)
save_plotly(visualize_det_on_graph_plotly(failure_structure_g, 'tanner', failure_structure_row['detector_events']), failure_structure_data_paths.output('cell_0030_1_html'))


In [ ]:
failure_structure_row = analysis_df[analysis_df['WL_hash'] == '509b0c29ecac32856bba94e0b68e0967'].iloc[0]
failure_structure_p = float(failure_structure_row['p'])
failure_structure_code = failure_structure_ca_codes[failure_structure_row['code']]
failure_structure_builder = CircuitBuilder(failure_structure_code)
failure_structure_circ = failure_structure_builder.get_circuit_level_circuit(error_rate=ErrorRate(p=failure_structure_p), d_rounds=3)
failure_structure_dem = failure_structure_circ.detector_error_model()
failure_structure_g, failure_structure_ = failure_structure_build_tanner_graph_from_dem(failure_structure_dem)
save_plotly(visualize_det_on_graph_plotly(failure_structure_g, 'tanner', failure_structure_row['detector_events']), failure_structure_data_paths.output('cell_0031_1_html'))


### Trapping Set

### Summarize candidate trapping parameters

Relate local trapping descriptors to the empirical case groups. Structural candidate membership and actual decoder failure remain separate properties.


In [ ]:
import ast
from tqdm.auto import tqdm

out_dir = failure_structure_data_paths.output('trapping_set')
out_dir.mkdir(parents=True, exist_ok=True)


In [ ]:
bfts_analysis_df = analysis_df.copy()
bfts_analysis_df = bfts_analysis_df.reset_index(drop=True)
bfts_analysis_df['case_id'] = np.arange(len(bfts_analysis_df))
cols = ['WL_hash', 'detector_events', 'active_detectors', 'induced_nodes', 'induced_edges', 'components', 'largest_component_size', 'cycle_rank', 'inverse_girth', 'density', 'mean_base_degree_on_active', 'trapping_param', 'error_node_parity', 'code', 'p']
trapping_values = bfts_analysis_df['trapping_param'].map(lambda x: ast.literal_eval(x) if isinstance(x, str) else tuple(x))
parity_values = bfts_analysis_df['error_node_parity'].map(lambda x: ast.literal_eval(x) if isinstance(x, str) else tuple(x))
bfts_analysis_df['trapping_a'] = trapping_values.map(lambda x: int(x[0]))
bfts_analysis_df['trapping_b'] = trapping_values.map(lambda x: int(x[1]))
bfts_analysis_df['error_even'] = parity_values.map(lambda x: int(x[0]))
bfts_analysis_df['error_odd'] = parity_values.map(lambda x: int(x[1]))

if failure_structure_config['print_level'] >= 2:
    print('number of cases:', len(bfts_analysis_df))
if failure_structure_config['print_level'] >= 2:
    print('unique WL_hash:', bfts_analysis_df['WL_hash'].nunique())

if failure_structure_config['print_level'] >= 2:
    print('unique trapping_param:', bfts_analysis_df['trapping_param'].nunique())
if failure_structure_config['print_level'] >= 2:
    print('unique error_node_parity:', bfts_analysis_df['error_node_parity'].nunique())
bfts_analysis_df.head()


In [ ]:
column_overview_rows = []
for failure_structure_col in tqdm(bfts_analysis_df.columns, desc='Trapping-set analysis', leave=True, disable=failure_structure_config['print_level'] == 1):
    s = bfts_analysis_df[failure_structure_col]
    column_overview_rows.append({
        'column': failure_structure_col,
        'dtype': str(s.dtype),
        'non_null': int(s.notna().sum()),
        'unique_values': int(s.nunique(dropna=True)),
        'sample_values': ', '.join(map(str, s.dropna().head(5).tolist())),
    })

column_overview_df = pd.DataFrame(column_overview_rows)
column_overview_df


#### Trapping-structure summaries

In [ ]:
trapping_group = bfts_analysis_df.groupby(['trapping_param', 'trapping_a', 'trapping_b'])
bfts_trapping_param_summary = trapping_group.agg(
    case_count=('case_id', 'size'),
    wl_hash_count=('WL_hash', 'nunique'),
    code_count=('code', 'nunique'),
    p_count=('p', 'nunique'),
    active_detectors_median=('active_detectors', 'median'),
    cycle_rank_median=('cycle_rank', 'median'),
    density_median=('density', 'median'),
    induced_nodes_median=('induced_nodes', 'median'),
    induced_edges_median=('induced_edges', 'median'),
).reset_index()
bfts_trapping_param_summary = bfts_trapping_param_summary.sort_values(['case_count', 'trapping_a', 'trapping_b'], ascending=[False, True, True]).reset_index(drop=True)

bfts_trapping_param_summary.to_csv(failure_structure_aggregate_path(failure_structure_data_paths.output('bfts_trapping_param_summary_csv')), index=False)
if failure_structure_config['print_level'] >= 2:
    print('saved:', failure_structure_aggregate_path(failure_structure_data_paths.output('bfts_trapping_param_summary_csv')))
bfts_trapping_param_summary.head(20)


In [ ]:
parity_group = bfts_analysis_df.groupby(['error_node_parity', 'error_even', 'error_odd'])
bfts_error_node_parity_summary = parity_group.agg(
    case_count=('case_id', 'size'),
    wl_hash_count=('WL_hash', 'nunique'),
    trapping_param_count=('trapping_param', 'nunique'),
    code_count=('code', 'nunique'),
    p_count=('p', 'nunique'),
    active_detectors_median=('active_detectors', 'median'),
    cycle_rank_median=('cycle_rank', 'median'),
    density_median=('density', 'median'),
).reset_index()
bfts_error_node_parity_summary = bfts_error_node_parity_summary.sort_values(['case_count', 'error_even', 'error_odd'], ascending=[False, True, True]).reset_index(drop=True)

bfts_error_node_parity_summary.to_csv(failure_structure_aggregate_path(failure_structure_data_paths.output('bfts_error_node_parity_summary_csv')), index=False)
if failure_structure_config['print_level'] >= 2:
    print('saved:', failure_structure_aggregate_path(failure_structure_data_paths.output('bfts_error_node_parity_summary_csv')))
bfts_error_node_parity_summary.head(20)


In [ ]:
wl_group = bfts_analysis_df.groupby('WL_hash')
bfts_wl_hash_summary = wl_group.agg(
    case_count=('case_id', 'size'),
    code_count=('code', 'nunique'),
    p_count=('p', 'nunique'),
    trapping_param_count=('trapping_param', 'nunique'),
    error_node_parity_count=('error_node_parity', 'nunique'),
    active_detectors_median=('active_detectors', 'median'),
    induced_nodes_median=('induced_nodes', 'median'),
    induced_edges_median=('induced_edges', 'median'),
    cycle_rank_median=('cycle_rank', 'median'),
    density_median=('density', 'median'),
    trapping_a_median=('trapping_a', 'median'),
    trapping_b_median=('trapping_b', 'median'),
    error_even_median=('error_even', 'median'),
    error_odd_median=('error_odd', 'median'),
).reset_index()
bfts_wl_hash_summary = bfts_wl_hash_summary.merge(wl_group['code'].unique().map(lambda x: ';'.join(map(str, sorted(x)))).rename('codes'), on='WL_hash')
bfts_wl_hash_summary = bfts_wl_hash_summary.merge(wl_group['p'].unique().map(lambda x: ';'.join(map(str, sorted(x)))).rename('p_values'), on='WL_hash')
bfts_wl_hash_summary = bfts_wl_hash_summary.merge(wl_group['trapping_param'].agg(lambda x: x.value_counts().index[0]).rename('dominant_trapping_param'), on='WL_hash')
bfts_wl_hash_summary = bfts_wl_hash_summary.merge(wl_group['error_node_parity'].agg(lambda x: x.value_counts().index[0]).rename('dominant_error_node_parity'), on='WL_hash')
bfts_wl_hash_summary = bfts_wl_hash_summary.sort_values(['case_count', 'trapping_param_count', 'cycle_rank_median'], ascending=[False, True, True]).reset_index(drop=True)

bfts_wl_hash_summary.to_csv(failure_structure_aggregate_path(failure_structure_data_paths.output('bfts_wl_hash_summary_csv')), index=False)
if failure_structure_config['print_level'] >= 2:
    print('saved:', failure_structure_aggregate_path(failure_structure_data_paths.output('bfts_wl_hash_summary_csv')))
bfts_wl_hash_summary.head(20)


one `WL_hash` -> multiple `trapping_param`

or

one `trapping_param` -> multiple `WL_hash`

In [ ]:
bfts_wl_trapping_cross_summary = bfts_analysis_df.groupby(['WL_hash', 'trapping_param', 'error_node_parity']).agg(
    case_count=('case_id', 'size'),
    code_count=('code', 'nunique'),
    p_count=('p', 'nunique'),
    active_detectors_median=('active_detectors', 'median'),
    cycle_rank_median=('cycle_rank', 'median'),
    density_median=('density', 'median'),
).reset_index().sort_values(['case_count', 'WL_hash'], ascending=[False, True]).reset_index(drop=True)
bfts_wl_trapping_cross_summary.to_csv(failure_structure_aggregate_path(failure_structure_data_paths.output('bfts_wl_trapping_cross_summary_csv')), index=False)
if failure_structure_config['print_level'] >= 2:
    print('saved:', failure_structure_aggregate_path(failure_structure_data_paths.output('bfts_wl_trapping_cross_summary_csv')))
bfts_wl_trapping_cross_summary.head(30)


In [ ]:
failure_structure_fig, axes = plt.subplots(2, 2, figsize=(16, 12))
trapping_top = bfts_trapping_param_summary.head(20).copy()
sns.barplot(
    data=trapping_top,
    x='case_count',
    y='trapping_param',
    ax=axes[0, 0],
)
axes[0, 0].set_title('Top trapping_param values')
axes[0, 0].set_xlabel('case_count')
axes[0, 0].set_ylabel('trapping_param = (error nodes, odd check nodes)')
parity_top = bfts_error_node_parity_summary.head(20).copy()
sns.barplot(
    data=parity_top,
    x='case_count',
    y='error_node_parity',
    ax=axes[0, 1],
)
axes[0, 1].set_title('Top error_node_parity values')
axes[0, 1].set_xlabel('case_count')
axes[0, 1].set_ylabel('error_node_parity = (even error nodes, odd error nodes)')
failure_structure_counts = bfts_analysis_df.groupby(['trapping_a', 'trapping_b']).size().reset_index(name='case_count').sort_values('case_count', ascending=False)
sns.scatterplot(
    data=failure_structure_counts,
    x='trapping_a',
    y='trapping_b',
    size='case_count',
    hue='case_count',
    sizes=(30, 300),
    ax=axes[1, 0],
)
axes[1, 0].set_title('Trapping parameter frequency')
axes[1, 0].set_xlabel('a = error nodes in trapping subgraph')
axes[1, 0].set_ylabel('b = odd-degree check nodes')
axes[1, 0].grid(True, ls='--', alpha=0.3)
sns.scatterplot(
    data=bfts_analysis_df,
    x='cycle_rank',
    y='trapping_b',
    hue='code',
    alpha=0.65,
    ax=axes[1, 1],
)
axes[1, 1].set_title('cycle_rank vs trapping_b')
axes[1, 1].set_xlabel('cycle_rank of induced subgraph')
axes[1, 1].set_ylabel('trapping_b')
axes[1, 1].grid(True, ls='--', alpha=0.3)
plt.tight_layout()

plt.savefig(failure_structure_data_paths.output('cell_0042_1_png'), bbox_inches='tight')
plt.show()
failure_structure_counts.head(20)


#### Classical-type trapping set candidates

`graph_det_features` already computed `trapping_param = (a, b)`, where:

- `a`: number of error nodes in the trapping subgraph;
- `b`: number of odd-degree check nodes in the trapping subgraph.

Frequent `WL_hash` with small `b` or small `b/a` are treated as stronger classical-type trapping-set candidates. `cycle_rank` and `density` are kept beside the candidate score because they help separate tree-like traps from cycle-rich traps.


In [ ]:
CLASSICAL_SMALL_B_THRESHOLD = 2
CLASSICAL_B_OVER_A_THRESHOLD = 0.2
CLASSICAL_RECURRENT_CASE_THRESHOLD = 5
CLASSICAL_CYCLE_RICH_THRESHOLD = 1
bfts_analysis_df['classical_b_over_a'] = np.where(bfts_analysis_df['trapping_a'] > 0, bfts_analysis_df['trapping_b'] / bfts_analysis_df['trapping_a'], np.nan)
bfts_analysis_df['classical_small_b'] = bfts_analysis_df['trapping_b'] <= CLASSICAL_SMALL_B_THRESHOLD
bfts_analysis_df['classical_low_ratio'] = bfts_analysis_df['classical_b_over_a'] <= CLASSICAL_B_OVER_A_THRESHOLD
bfts_analysis_df['classical_cycle_rich'] = bfts_analysis_df['cycle_rank'] >= CLASSICAL_CYCLE_RICH_THRESHOLD
trapping_case_counts = bfts_analysis_df['trapping_param'].value_counts()
bfts_analysis_df['classical_recurrent_trapping_param'] = bfts_analysis_df['trapping_param'].map(trapping_case_counts) >= CLASSICAL_RECURRENT_CASE_THRESHOLD
bfts_analysis_df['classical_candidate'] = bfts_analysis_df['classical_recurrent_trapping_param'] & (bfts_analysis_df['classical_small_b'] | bfts_analysis_df['classical_low_ratio'])
bfts_classical_trapping_candidates = bfts_analysis_df.groupby(['trapping_param', 'trapping_a', 'trapping_b']).agg(
    case_count=('case_id', 'size'),
    wl_hash_count=('WL_hash', 'nunique'),
    code_count=('code', 'nunique'),
    p_count=('p', 'nunique'),
    candidate_case_count=('classical_candidate', 'sum'),
    candidate_fraction=('classical_candidate', 'mean'),
    b_over_a_mean=('classical_b_over_a', 'mean'),
    b_over_a_median=('classical_b_over_a', 'median'),
    active_detectors_median=('active_detectors', 'median'),
    cycle_rank_median=('cycle_rank', 'median'),
    cycle_rich_fraction=('classical_cycle_rich', 'mean'),
    density_median=('density', 'median'),
    error_odd_median=('error_odd', 'median'),
).reset_index().sort_values(['candidate_case_count', 'case_count', 'trapping_b', 'b_over_a_median'], ascending=[False, False, True, True]).reset_index(drop=True)
bfts_classical_trapping_by_wl_hash = bfts_analysis_df.groupby('WL_hash').agg(
    case_count=('case_id', 'size'),
    trapping_param_count=('trapping_param', 'nunique'),
    candidate_case_count=('classical_candidate', 'sum'),
    candidate_fraction=('classical_candidate', 'mean'),
    b_over_a_median=('classical_b_over_a', 'median'),
    trapping_a_median=('trapping_a', 'median'),
    trapping_b_median=('trapping_b', 'median'),
    cycle_rank_median=('cycle_rank', 'median'),
    density_median=('density', 'median'),
    code_count=('code', 'nunique'),
    p_count=('p', 'nunique'),
).reset_index()
bfts_classical_trapping_by_wl_hash = bfts_classical_trapping_by_wl_hash.merge(bfts_analysis_df.groupby('WL_hash')['trapping_param'].agg(lambda x: x.value_counts().index[0]).rename('dominant_trapping_param'), on='WL_hash')
bfts_classical_trapping_by_wl_hash = bfts_classical_trapping_by_wl_hash.sort_values(['candidate_case_count', 'case_count', 'trapping_param_count'], ascending=[False, False, True]).reset_index(drop=True)

bfts_classical_trapping_candidates.to_csv(failure_structure_data_paths.output('bfts_classical_trapping_candidates_csv'), index=False)
bfts_classical_trapping_by_wl_hash.to_csv(failure_structure_aggregate_path(failure_structure_data_paths.output('bfts_classical_trapping_by_wl_hash_csv')), index=False)
if failure_structure_config['print_level'] >= 2:
    print('saved:', failure_structure_data_paths.output('bfts_classical_trapping_candidates_csv'))

if failure_structure_config['print_level'] >= 2:
    print('saved:', failure_structure_aggregate_path(failure_structure_data_paths.output('bfts_classical_trapping_by_wl_hash_csv')))
bfts_classical_trapping_candidates.head(20)


In [ ]:
failure_structure_fig, axes = plt.subplots(1, 3, figsize=(18, 5))
classical_top = bfts_classical_trapping_candidates.head(20).copy()
sns.barplot(
    data=classical_top,
    x='case_count',
    y='trapping_param',
    hue='candidate_fraction',
    ax=axes[0],
)
axes[0].set_title('Classical-type candidates by (a, b)')
axes[0].set_xlabel('case_count')
axes[0].set_ylabel('trapping_param')
sns.scatterplot(
    data=bfts_classical_trapping_candidates,
    x='trapping_a',
    y='trapping_b',
    size='case_count',
    hue='candidate_fraction',
    sizes=(30, 260),
    ax=axes[1],
)
axes[1].axhline(CLASSICAL_SMALL_B_THRESHOLD, color='black', lw=1, ls='--')
axes[1].set_title('Classical (a, b) candidate map')
axes[1].set_xlabel('a = error nodes')
axes[1].set_ylabel('b = odd-degree check nodes')
axes[1].grid(True, ls='--', alpha=0.3)
sns.scatterplot(
    data=bfts_analysis_df,
    x='cycle_rank',
    y='classical_b_over_a',
    hue='classical_candidate',
    alpha=0.65,
    ax=axes[2],
)
axes[2].axhline(CLASSICAL_B_OVER_A_THRESHOLD, color='black', lw=1, ls='--')
axes[2].set_title('Cycle rank vs b/a')
axes[2].set_xlabel('cycle_rank')
axes[2].set_ylabel('b / a')
axes[2].grid(True, ls='--', alpha=0.3)
plt.tight_layout()

plt.savefig(failure_structure_data_paths.output('cell_0046_1_png'), bbox_inches='tight')
plt.show()
bfts_classical_trapping_by_wl_hash.head(20)


#### Symmetric stabilizer trapping set candidates

This part looks for a different kind of candidate structure: active detector patterns that are close to mirror-symmetric in detector-coordinate space. The motivation is that a near-symmetric set of active stabilizer measurements can indicate two locally similar explanations for the same syndrome pattern.

The only graph-related object rebuilt here is the circuit/DEM coordinate map for each `(code, p)` pair; no graph features are recomputed.

Two mirror scores are computed:

- `mirror_x_jaccard`: active detector set compared with its x-axis mirror image;
- `mirror_y_jaccard`: active detector set compared with its y-axis mirror image.

The larger of the two is `best_mirror_jaccard`. High values mark symmetric stabilizer trapping-set candidates.


In [ ]:
SYMMETRY_JACCARD_THRESHOLD = 0.6
SYMMETRY_RECURRENT_CASE_THRESHOLD = 3
coord_mirror_cache = {}
case_symmetry_rows = []

for (code_name, failure_structure_p), group in tqdm(bfts_analysis_df.groupby(['code', 'p']), desc='Build detector-coordinate mirror maps'):
    cache_key = (code_name, float(failure_structure_p))
    if cache_key not in coord_mirror_cache:
        failure_structure_code = failure_structure_ca_codes[code_name]
        failure_structure_builder = CircuitBuilder(failure_structure_code)
        failure_structure_circ = failure_structure_builder.get_circuit_level_circuit(error_rate=ErrorRate(p=float(failure_structure_p)), d_rounds=3)
        failure_structure_dem = failure_structure_circ.detector_error_model()
        coords = {int(k): tuple(v) for k, v in failure_structure_dem.get_detector_coordinates().items()}
        xs = [coord[0] for coord in coords.values()]
        ys = [coord[1] for coord in coords.values()]
        x_sum = min(xs) + max(xs)
        y_sum = min(ys) + max(ys)
        coord_to_det = {coord: det for det, coord in coords.items()}
        mirror_x = {}
        mirror_y = {}

        for det, coord in tqdm(coords.items(), desc='Trapping-set analysis', leave=False, disable=failure_structure_config['print_level'] == 1):
            mirrored_x_coord = (x_sum - coord[0], coord[1], *coord[2:])
            mirrored_y_coord = (coord[0], y_sum - coord[1], *coord[2:])
            if mirrored_x_coord in coord_to_det:
                mirror_x[det] = coord_to_det[mirrored_x_coord]

            if mirrored_y_coord in coord_to_det:
                mirror_y[det] = coord_to_det[mirrored_y_coord]

        coord_mirror_cache[cache_key] = {'mirror_x': mirror_x, 'mirror_y': mirror_y}

    mirror_x = coord_mirror_cache[cache_key]['mirror_x']
    mirror_y = coord_mirror_cache[cache_key]['mirror_y']
    for failure_structure_, failure_structure_row in tqdm(group.iterrows(), desc='Trapping-set analysis', leave=False, disable=failure_structure_config['print_level'] == 1):
        active_dets = set(event_to_active_ids(failure_structure_row['detector_events']))
        mirrored_x = {mirror_x[d] for d in active_dets if d in mirror_x}
        mirrored_y = {mirror_y[d] for d in active_dets if d in mirror_y}
        union_x = active_dets | mirrored_x
        union_y = active_dets | mirrored_y
        mirror_x_jaccard = len(active_dets & mirrored_x) / len(union_x) if len(union_x) else 1.0
        mirror_y_jaccard = len(active_dets & mirrored_y) / len(union_y) if len(union_y) else 1.0

        if mirror_x_jaccard >= mirror_y_jaccard:
            best_axis = 'x'
            best_score = mirror_x_jaccard
            best_unmatched = len(active_dets - mirrored_x)
        else:
            best_axis = 'y'
            best_score = mirror_y_jaccard
            best_unmatched = len(active_dets - mirrored_y)

        case_symmetry_rows.append({
            'case_id': int(failure_structure_row['case_id']),
            'WL_hash': failure_structure_row['WL_hash'],
            'trapping_param': failure_structure_row['trapping_param'],
            'error_node_parity': failure_structure_row['error_node_parity'],
            'code': failure_structure_row['code'],
            'p': float(failure_structure_row['p']),
            'mirror_x_jaccard': mirror_x_jaccard,
            'mirror_y_jaccard': mirror_y_jaccard,
            'best_mirror_axis': best_axis,
            'best_mirror_jaccard': best_score,
            'best_mirror_unmatched_active': best_unmatched,
            'symmetric_stabilizer_candidate': best_score >= SYMMETRY_JACCARD_THRESHOLD,
        })

bfts_symmetry_by_case = pd.DataFrame(case_symmetry_rows)
bfts_analysis_df = bfts_analysis_df.merge(bfts_symmetry_by_case[['case_id', 'mirror_x_jaccard', 'mirror_y_jaccard', 'best_mirror_axis', 'best_mirror_jaccard', 'best_mirror_unmatched_active', 'symmetric_stabilizer_candidate']], on='case_id', how='left')
bfts_symmetric_stabilizer_by_wl_hash = bfts_analysis_df.groupby('WL_hash').agg(
    case_count=('case_id', 'size'),
    symmetric_case_count=('symmetric_stabilizer_candidate', 'sum'),
    symmetric_fraction=('symmetric_stabilizer_candidate', 'mean'),
    best_mirror_jaccard_mean=('best_mirror_jaccard', 'mean'),
    best_mirror_jaccard_max=('best_mirror_jaccard', 'max'),
    best_mirror_unmatched_median=('best_mirror_unmatched_active', 'median'),
    trapping_param_count=('trapping_param', 'nunique'),
    trapping_b_median=('trapping_b', 'median'),
    cycle_rank_median=('cycle_rank', 'median'),
    code_count=('code', 'nunique'),
    p_count=('p', 'nunique'),
).reset_index()
bfts_symmetric_stabilizer_by_wl_hash = bfts_symmetric_stabilizer_by_wl_hash.merge(bfts_analysis_df.groupby('WL_hash')['best_mirror_axis'].agg(lambda x: x.value_counts().index[0]).rename('dominant_mirror_axis'), on='WL_hash')
bfts_symmetric_stabilizer_by_wl_hash = bfts_symmetric_stabilizer_by_wl_hash.merge(bfts_analysis_df.groupby('WL_hash')['trapping_param'].agg(lambda x: x.value_counts().index[0]).rename('dominant_trapping_param'), on='WL_hash')
bfts_symmetric_stabilizer_by_wl_hash = bfts_symmetric_stabilizer_by_wl_hash.sort_values(['symmetric_case_count', 'best_mirror_jaccard_max', 'case_count'], ascending=[False, False, False]).reset_index(drop=True)
bfts_symmetric_stabilizer_candidates = bfts_symmetric_stabilizer_by_wl_hash[bfts_symmetric_stabilizer_by_wl_hash['symmetric_case_count'] >= SYMMETRY_RECURRENT_CASE_THRESHOLD].copy()

bfts_symmetric_stabilizer_by_wl_hash.to_csv(failure_structure_aggregate_path(failure_structure_data_paths.output('bfts_symmetric_stabilizer_by_wl_hash_csv')), index=False)
bfts_symmetric_stabilizer_candidates.to_csv(failure_structure_data_paths.output('bfts_symmetric_stabilizer_candidates_csv'), index=False)
if failure_structure_config['print_level'] >= 2:
    print('saved:', failure_structure_aggregate_path(failure_structure_data_paths.output('bfts_symmetric_stabilizer_by_wl_hash_csv')))

if failure_structure_config['print_level'] >= 2:
    print('saved:', failure_structure_data_paths.output('bfts_symmetric_stabilizer_candidates_csv'))
bfts_symmetric_stabilizer_candidates.head(20)


In [ ]:
failure_structure_fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.histplot(
    data=bfts_analysis_df,
    x='best_mirror_jaccard',
    bins=20,
    ax=axes[0],
)
axes[0].axvline(SYMMETRY_JACCARD_THRESHOLD, color='black', lw=1, ls='--')
axes[0].set_title('Best mirror symmetry score')
axes[0].set_xlabel('best_mirror_jaccard')
axes[0].set_ylabel('case_count')
sns.scatterplot(
    data=bfts_analysis_df,
    x='best_mirror_jaccard',
    y='trapping_b',
    hue='symmetric_stabilizer_candidate',
    alpha=0.65,
    ax=axes[1],
)
axes[1].axvline(SYMMETRY_JACCARD_THRESHOLD, color='black', lw=1, ls='--')
axes[1].set_title('Symmetry score vs trapping_b')
axes[1].set_xlabel('best_mirror_jaccard')
axes[1].set_ylabel('trapping_b')
axes[1].grid(True, ls='--', alpha=0.3)
sym_top = bfts_symmetric_stabilizer_by_wl_hash.head(20).copy()
sns.barplot(
    data=sym_top,
    x='symmetric_case_count',
    y='WL_hash',
    hue='dominant_mirror_axis',
    ax=axes[2],
)
axes[2].set_title('Top symmetric WL_hash candidates')
axes[2].set_xlabel('symmetric_case_count')
axes[2].set_ylabel('WL_hash')
plt.tight_layout()

plt.savefig(failure_structure_data_paths.output('cell_0049_1_png'), bbox_inches='tight')
plt.show()
bfts_symmetric_stabilizer_by_wl_hash.head(20)


### Feature Distance

### Compare feature-space neighborhoods

Use the recorded structural descriptors to compare nearby groups. Similarity in standardized feature space need not imply identical graph connectivity or decoder behavior.


In [ ]:
TOP_K_SIGNATURES_FOR_DISTANCE = 30
FEATURE_DISTANCE_THRESHOLD = 2.0


In [ ]:
distance_feature_cols = ['active_detectors', 'induced_nodes', 'induced_edges', 'components', 'largest_component_size', 'cycle_rank', 'inverse_girth', 'density', 'mean_base_degree_on_active', 'trapping_a', 'trapping_b', 'error_even', 'error_odd']
top_hashes_for_distance = bfts_wl_hash_summary.head(TOP_K_SIGNATURES_FOR_DISTANCE)['WL_hash'].tolist()
wl_feature_profile = bfts_analysis_df[bfts_analysis_df['WL_hash'].isin(top_hashes_for_distance)].groupby('WL_hash')[distance_feature_cols].mean().loc[top_hashes_for_distance]
X = wl_feature_profile.to_numpy(dtype=float)
feature_mean = X.mean(axis=0)
feature_std = X.std(axis=0)
feature_std[feature_std == 0] = 1.0
X_scaled = (X - feature_mean) / feature_std
feature_distance_rows = []

for failure_structure_i, j in tqdm(list(combinations(range(len(top_hashes_for_distance)), 2)), desc='Pairwise WL_hash feature distance'):
    h1 = top_hashes_for_distance[failure_structure_i]
    h2 = top_hashes_for_distance[j]
    dist = float(np.linalg.norm(X_scaled[failure_structure_i] - X_scaled[j]))
    feature_distance_rows.append({'WL_hash_1': h1, 'WL_hash_2': h2, 'feature_distance': dist})

for failure_structure_i, h in tqdm(enumerate(top_hashes_for_distance), desc='Trapping-set analysis', leave=True, disable=failure_structure_config['print_level'] == 1):
    feature_distance_rows.append({'WL_hash_1': h, 'WL_hash_2': h, 'feature_distance': 0.0})
bfts_wl_feature_distance = pd.DataFrame(feature_distance_rows)

bfts_wl_feature_distance.to_csv(failure_structure_data_paths.output('bfts_wl_feature_distance_csv'), index=False)
if failure_structure_config['print_level'] >= 2:
    print('saved:', failure_structure_data_paths.output('bfts_wl_feature_distance_csv'))
bfts_wl_feature_distance.sort_values('feature_distance', ascending=False).head(30)


In [ ]:
feature_distance_matrix = pd.DataFrame(np.nan, index=top_hashes_for_distance, columns=top_hashes_for_distance, dtype=float)
for failure_structure_, failure_structure_row in tqdm(bfts_wl_feature_distance.iterrows(), desc='Trapping-set analysis', leave=True, disable=failure_structure_config['print_level'] == 1):
    h1 = failure_structure_row['WL_hash_1']
    h2 = failure_structure_row['WL_hash_2']
    dist = failure_structure_row['feature_distance']
    feature_distance_matrix.loc[h1, h2] = dist
    feature_distance_matrix.loc[h2, h1] = dist

failure_structure_fig, failure_structure_ax = plt.subplots(figsize=(12, 10))
sns.heatmap(feature_distance_matrix, cmap='viridis_r', ax=failure_structure_ax)
failure_structure_ax.set_title(f'Feature-space distance among top {len(top_hashes_for_distance)} WL_hash groups')
failure_structure_ax.set_xlabel('WL_hash')
failure_structure_ax.set_ylabel('WL_hash')
plt.tight_layout()

plt.savefig(failure_structure_data_paths.output('cell_0054_1_png'), bbox_inches='tight')
plt.show()
bfts_wl_feature_distance[bfts_wl_feature_distance['WL_hash_1'] != bfts_wl_feature_distance['WL_hash_2']].sort_values('feature_distance').head(20)


#### Feature distance clusters

Two `WL_hash` group are connected when their standardized feature distance is no larger than `FEATURE_DISTANCE_THRESHOLD`.

In [ ]:
finite_feature_pairs = bfts_wl_feature_distance[bfts_wl_feature_distance['WL_hash_1'] != bfts_wl_feature_distance['WL_hash_2']].copy()
feature_similarity_graph = nx.Graph()
feature_similarity_graph.add_nodes_from(top_hashes_for_distance)

for failure_structure_, failure_structure_row in tqdm(finite_feature_pairs.iterrows(), desc='Trapping-set analysis', leave=True, disable=failure_structure_config['print_level'] == 1):
    if failure_structure_row['feature_distance'] <= FEATURE_DISTANCE_THRESHOLD:
        feature_similarity_graph.add_edge(failure_structure_row['WL_hash_1'], failure_structure_row['WL_hash_2'], feature_distance=float(failure_structure_row['feature_distance']))

cluster_rows = []
for cluster_id, component in tqdm(enumerate(nx.connected_components(feature_similarity_graph)), desc='Trapping-set analysis', leave=True, disable=failure_structure_config['print_level'] == 1):
    component = sorted(component, key=lambda h: top_hashes_for_distance.index(h))
    class_rows = bfts_wl_hash_summary[bfts_wl_hash_summary['WL_hash'].isin(component)]
    mean_feature_distance = 0.0 if len(component) <= 1 else finite_feature_pairs[finite_feature_pairs['WL_hash_1'].isin(component) & finite_feature_pairs['WL_hash_2'].isin(component)]['feature_distance'].mean()
    cluster_rows.append({
        'feature_cluster_id': cluster_id,
        'num_wl_hash_groups': len(component),
        'total_cases': int(class_rows['case_count'].sum()),
        'WL_hashes': ';'.join(component),
        'case_count_by_hash': ';'.join((f'{r.WL_hash}:{int(r.case_count)}' for r in class_rows.itertuples())),
        'mean_feature_distance': float(mean_feature_distance),
    })

bfts_feature_distance_clusters = pd.DataFrame(cluster_rows).sort_values(['num_wl_hash_groups', 'total_cases'], ascending=[False, False]).reset_index(drop=True)
bfts_feature_distance_clusters.to_csv(failure_structure_data_paths.output('bfts_feature_distance_clusters_csv'), index=False)
if failure_structure_config['print_level'] >= 2:
    print('saved:', failure_structure_data_paths.output('bfts_feature_distance_clusters_csv'))
bfts_feature_distance_clusters.head(20)


### Graph Edit Distance

### Compare graph-edit neighborhoods

Inspect graph-edit proximity among the selected representative hashes. The restricted representative set and any unavailable distances constrain what these clusters can establish.


In [ ]:
TOP_K_HASHES_FOR_GED = 10
GED_TIMEOUT_SECONDS = 1.0
GED_SIMILARITY_THRESHOLD = 2.0


In [ ]:
top_hashes_for_ged = bfts_wl_hash_summary.head(TOP_K_HASHES_FOR_GED)['WL_hash'].tolist()
representative_case_ids = bfts_analysis_df[bfts_analysis_df['WL_hash'].isin(top_hashes_for_ged)].sort_values(['WL_hash', 'case_id']).drop_duplicates('WL_hash').set_index('WL_hash')['case_id'].loc[top_hashes_for_ged]
representative_graphs = {}
tanner_graph_cache = {}

for wl_hash, case_id in tqdm(representative_case_ids.items(), total=len(representative_case_ids), desc='Materialize representative induced subgraphs'):
    failure_structure_row = bfts_analysis_df.loc[int(case_id)]
    graph_key = (failure_structure_row['code'], float(failure_structure_row['p']))
    if graph_key not in tanner_graph_cache:
        failure_structure_code = failure_structure_ca_codes[failure_structure_row['code']]
        failure_structure_builder = CircuitBuilder(failure_structure_code)
        failure_structure_circ = failure_structure_builder.get_circuit_level_circuit(error_rate=ErrorRate(p=float(failure_structure_row['p'])), d_rounds=3)
        failure_structure_dem = failure_structure_circ.detector_error_model()
        failure_structure_g, failure_structure_ = failure_structure_build_tanner_graph_from_dem(failure_structure_dem)
        tanner_graph_cache[graph_key] = failure_structure_g

    active_dets = set(event_to_active_ids(failure_structure_row['detector_events']))
    representative_graphs[wl_hash] = induced_subgraph(tanner_graph_cache[graph_key], active_dets)

ged_rows = []
for h1, h2 in tqdm(list(combinations(top_hashes_for_ged, 2)), desc='Pairwise representative GED'):
    ged = nx.graph_edit_distance(
        representative_graphs[h1],
        representative_graphs[h2],
        node_match=lambda left_attrs,
        right_attrs: left_attrs.get('kind') == right_attrs.get('kind') and bool(left_attrs.get('active', False)) == bool(right_attrs.get('active', False)),
        timeout=GED_TIMEOUT_SECONDS,
    )
    ged_rows.append({
        'WL_hash_1': h1,
        'WL_hash_2': h2,
        'ged': np.nan if ged is None else float(ged),
        'timeout_seconds': GED_TIMEOUT_SECONDS,
    })

for h in tqdm(top_hashes_for_ged, desc='Trapping-set analysis', leave=True, disable=failure_structure_config['print_level'] == 1):
    ged_rows.append({
        'WL_hash_1': h,
        'WL_hash_2': h,
        'ged': 0.0,
        'timeout_seconds': GED_TIMEOUT_SECONDS,
    })

bfts_representative_ged = pd.DataFrame(ged_rows)
bfts_representative_ged.to_csv(failure_structure_data_paths.output('bfts_representative_ged_csv'), index=False)
if failure_structure_config['print_level'] >= 2:
    print('saved:', failure_structure_data_paths.output('bfts_representative_ged_csv'))
bfts_representative_ged.sort_values('ged').head(20)


In [ ]:
ged_matrix = pd.DataFrame(np.nan, index=top_hashes_for_ged, columns=top_hashes_for_ged, dtype=float)
for failure_structure_, failure_structure_row in tqdm(bfts_representative_ged.iterrows(), desc='Trapping-set analysis', leave=True, disable=failure_structure_config['print_level'] == 1):
    h1 = failure_structure_row['WL_hash_1']
    h2 = failure_structure_row['WL_hash_2']
    ged = failure_structure_row['ged']
    ged_matrix.loc[h1, h2] = ged
    ged_matrix.loc[h2, h1] = ged

failure_structure_fig, failure_structure_ax = plt.subplots(figsize=(12, 10))
sns.heatmap(ged_matrix, cmap='viridis_r', ax=failure_structure_ax)
failure_structure_ax.set_title(f'Representative GED among top {len(top_hashes_for_ged)} WL_hash groups')
failure_structure_ax.set_xlabel('WL_hash')
failure_structure_ax.set_ylabel('WL_hash')
plt.tight_layout()

plt.savefig(failure_structure_data_paths.output('cell_0060_1_png'), bbox_inches='tight')
plt.show()
finite_ged_pairs = bfts_representative_ged[(bfts_representative_ged['WL_hash_1'] != bfts_representative_ged['WL_hash_2']) & bfts_representative_ged['ged'].notna()].copy()
finite_ged_pairs.sort_values('ged').head(20)


#### GED-near WL_hash clusters


In [ ]:
finite_ged_pairs = bfts_representative_ged[(bfts_representative_ged['WL_hash_1'] != bfts_representative_ged['WL_hash_2']) & bfts_representative_ged['ged'].notna()].copy()
ged_similarity_graph = nx.Graph()
ged_similarity_graph.add_nodes_from(top_hashes_for_ged)

for failure_structure_, failure_structure_row in tqdm(finite_ged_pairs.iterrows(), desc='Trapping-set analysis', leave=True, disable=failure_structure_config['print_level'] == 1):
    if failure_structure_row['ged'] <= GED_SIMILARITY_THRESHOLD:
        ged_similarity_graph.add_edge(failure_structure_row['WL_hash_1'], failure_structure_row['WL_hash_2'], ged=float(failure_structure_row['ged']))

ged_cluster_rows = []
for cluster_id, component in tqdm(enumerate(nx.connected_components(ged_similarity_graph)), desc='Trapping-set analysis', leave=True, disable=failure_structure_config['print_level'] == 1):
    component = sorted(component, key=lambda h: top_hashes_for_ged.index(h))
    class_rows = bfts_wl_hash_summary[bfts_wl_hash_summary['WL_hash'].isin(component)]
    mean_pairwise_ged = 0.0 if len(component) <= 1 else finite_ged_pairs[finite_ged_pairs['WL_hash_1'].isin(component) & finite_ged_pairs['WL_hash_2'].isin(component)]['ged'].mean()
    ged_cluster_rows.append({
        'ged_cluster_id': cluster_id,
        'num_wl_hash_groups': len(component),
        'total_cases': int(class_rows['case_count'].sum()),
        'WL_hashes': ';'.join(component),
        'case_count_by_hash': ';'.join((f'{r.WL_hash}:{int(r.case_count)}' for r in class_rows.itertuples())),
        'mean_pairwise_ged': float(mean_pairwise_ged),
    })

bfts_ged_similarity_clusters = pd.DataFrame(ged_cluster_rows).sort_values(['num_wl_hash_groups', 'total_cases'], ascending=[False, False]).reset_index(drop=True)
bfts_ged_similarity_clusters.to_csv(failure_structure_data_paths.output('bfts_ged_similarity_clusters_csv'), index=False)
if failure_structure_config['print_level'] >= 2:
    print('saved:', failure_structure_data_paths.output('bfts_ged_similarity_clusters_csv'))
bfts_ged_similarity_clusters.head(20)


## Adversarial Trapping Set Testing

Construct certain detection events (conforming to a trapping set structure), and then use these events to judge two decoders.

In [ ]:
import csv
import json
from src.decoders import BPOSD as failure_structure_bposd, Tesseract

ADVERSARIAL_MAX_SEED_EVENTS = 24
ADVERSARIAL_MAX_GENERATED_EVENTS = 60
ADVERSARIAL_MAX_SEED_EVENTS_PER_GROUP = 5
ADVERSARIAL_MAX_ACTIVE_DETECTORS = 24
ADVERSARIAL_BPOSD_MAX_ITER = 60
ADVERSARIAL_BPOSD_OSD_ORDER = 10
ADVERSARIAL_TESSERACT_BEAM = 5
ADVERSARIAL_TESSERACT_PQ_LIMIT = 500
ADVERSARIAL_TESSERACT_SEED = 0
seed_pool = bfts_analysis_df.copy()
seed_pool['classical_candidate'] = seed_pool['classical_candidate'].fillna(False).astype(bool)
seed_pool['symmetric_stabilizer_candidate'] = seed_pool['symmetric_stabilizer_candidate'].fillna(False).astype(bool)
seed_pool = seed_pool[(seed_pool['classical_candidate'] | seed_pool['symmetric_stabilizer_candidate']) & (seed_pool['active_detectors'] > 0) & (seed_pool['active_detectors'] <= ADVERSARIAL_MAX_ACTIVE_DETECTORS)].copy()
assert len(seed_pool) > 0, 'No trapping-set candidate seed events are available under the current thresholds.'
seed_pool['adversarial_seed_score'] = 3 * seed_pool['classical_candidate'].astype(int) + 3 * seed_pool['symmetric_stabilizer_candidate'].astype(int) + seed_pool['cycle_rank'].astype(float) + seed_pool['active_detectors'].astype(float) / 100.0
seed_pool = seed_pool.sort_values(['adversarial_seed_score', 'active_detectors'], ascending=[False, False]).reset_index(drop=True)
adversarial_event_rows = []
seen_adversarial_events = set()

for failure_structure_, failure_structure_row in tqdm(seed_pool.head(ADVERSARIAL_MAX_SEED_EVENTS).iterrows(), desc='Trapping-set analysis', leave=True, disable=failure_structure_config['print_level'] == 1):
    event_bits = failure_structure_row['detector_events']
    event_key = (failure_structure_row['code'], float(failure_structure_row['p']), event_bits)
    if event_key in seen_adversarial_events:
        continue
    seen_adversarial_events.add(event_key)
    adversarial_event_rows.append({
        'construction': 'original_trapping_candidate',
        'code': failure_structure_row['code'],
        'p': float(failure_structure_row['p']),
        'detector_events': event_bits,
        'source_case_ids': str(int(failure_structure_row['case_id'])),
        'source_WL_hashes': failure_structure_row['WL_hash'],
        'source_trapping_params': failure_structure_row['trapping_param'],
        'source_classical_candidate': bool(failure_structure_row['classical_candidate']),
        'source_symmetric_candidate': bool(failure_structure_row['symmetric_stabilizer_candidate']),
    })

generated_count = 0
for (code_name, failure_structure_p, trapping_param), grp in tqdm(seed_pool.groupby(['code', 'p', 'trapping_param']), desc='Trapping-set analysis', leave=True, disable=failure_structure_config['print_level'] == 1):
    if generated_count >= ADVERSARIAL_MAX_GENERATED_EVENTS:
        break
    grp = grp.drop_duplicates('detector_events').head(ADVERSARIAL_MAX_SEED_EVENTS_PER_GROUP).reset_index(drop=True)

    if len(grp) < 2:
        continue
    for left_idx, right_idx in tqdm(combinations(range(len(grp)), 2), desc='Trapping-set analysis', leave=False, disable=failure_structure_config['print_level'] == 1):
        if generated_count >= ADVERSARIAL_MAX_GENERATED_EVENTS:
            break
        left = grp.loc[left_idx]
        right = grp.loc[right_idx]
        left_bits = np.fromiter((ch == '1' for ch in left['detector_events']), dtype=np.uint8)
        right_bits = np.fromiter((ch == '1' for ch in right['detector_events']), dtype=np.uint8)
        mixed_bits = np.bitwise_xor(left_bits, right_bits)
        active_count = int(mixed_bits.sum())

        if active_count == 0 or active_count > ADVERSARIAL_MAX_ACTIVE_DETECTORS:
            continue
        event_bits = ''.join(mixed_bits.astype(str))
        event_key = (code_name, float(failure_structure_p), event_bits)

        if event_key in seen_adversarial_events:
            continue
        seen_adversarial_events.add(event_key)
        generated_count += 1
        adversarial_event_rows.append({
            'construction': 'xor_same_trapping_param',
            'code': code_name,
            'p': float(failure_structure_p),
            'detector_events': event_bits,
            'source_case_ids': f"{int(left['case_id'])};{int(right['case_id'])}",
            'source_WL_hashes': f"{left['WL_hash']};{right['WL_hash']}",
            'source_trapping_params': trapping_param,
            'source_classical_candidate': bool(left['classical_candidate'] or right['classical_candidate']),
            'source_symmetric_candidate': bool(left['symmetric_stabilizer_candidate'] or right['symmetric_stabilizer_candidate']),
        })

for (code_name, failure_structure_p, wl_hash), grp in tqdm(seed_pool.groupby(['code', 'p', 'WL_hash']), desc='Trapping-set analysis', leave=True, disable=failure_structure_config['print_level'] == 1):
    if generated_count >= ADVERSARIAL_MAX_GENERATED_EVENTS:
        break
    grp = grp.drop_duplicates('detector_events').head(ADVERSARIAL_MAX_SEED_EVENTS_PER_GROUP).reset_index(drop=True)

    if len(grp) < 2:
        continue
    for left_idx, right_idx in tqdm(combinations(range(len(grp)), 2), desc='Trapping-set analysis', leave=False, disable=failure_structure_config['print_level'] == 1):
        if generated_count >= ADVERSARIAL_MAX_GENERATED_EVENTS:
            break
        left = grp.loc[left_idx]
        right = grp.loc[right_idx]
        left_bits = np.fromiter((ch == '1' for ch in left['detector_events']), dtype=np.uint8)
        right_bits = np.fromiter((ch == '1' for ch in right['detector_events']), dtype=np.uint8)
        mixed_bits = np.bitwise_or(left_bits, right_bits)
        active_count = int(mixed_bits.sum())

        if active_count == 0 or active_count > ADVERSARIAL_MAX_ACTIVE_DETECTORS:
            continue
        event_bits = ''.join(mixed_bits.astype(str))
        event_key = (code_name, float(failure_structure_p), event_bits)

        if event_key in seen_adversarial_events:
            continue
        seen_adversarial_events.add(event_key)
        generated_count += 1
        adversarial_event_rows.append({
            'construction': 'union_same_WL_hash',
            'code': code_name,
            'p': float(failure_structure_p),
            'detector_events': event_bits,
            'source_case_ids': f"{int(left['case_id'])};{int(right['case_id'])}",
            'source_WL_hashes': wl_hash,
            'source_trapping_params': f"{left['trapping_param']};{right['trapping_param']}",
            'source_classical_candidate': bool(left['classical_candidate'] or right['classical_candidate']),
            'source_symmetric_candidate': bool(left['symmetric_stabilizer_candidate'] or right['symmetric_stabilizer_candidate']),
        })

assert len(adversarial_event_rows) > 0, 'No adversarial detection events were constructed.'
graph_cache = {}
validated_adversarial_rows = []

for failure_structure_row in tqdm(adversarial_event_rows, desc='Validating constructed trapping-set events'):
    cache_key = (failure_structure_row['code'], float(failure_structure_row['p']))
    if cache_key not in graph_cache:
        failure_structure_code = failure_structure_ca_codes[failure_structure_row['code']]
        failure_structure_builder = CircuitBuilder(failure_structure_code)
        failure_structure_circ = failure_structure_builder.get_circuit_level_circuit(error_rate=ErrorRate(p=float(failure_structure_row['p'])), d_rounds=3)
        failure_structure_dem = failure_structure_circ.detector_error_model()
        failure_structure_g, failure_structure_ = failure_structure_build_tanner_graph_from_dem(failure_structure_dem)
        graph_cache[cache_key] = (failure_structure_g, failure_structure_dem)

    failure_structure_g, failure_structure_dem = graph_cache[cache_key]
    assert len(failure_structure_row['detector_events']) == failure_structure_dem.num_detectors, (failure_structure_row['code'], failure_structure_row['p'], len(failure_structure_row['detector_events']), failure_structure_dem.num_detectors)
    feats = graph_det_features(failure_structure_g, failure_structure_row['detector_events'])
    trapping_a, trapping_b = feats['trapping_param']
    trapping_a = int(trapping_a)
    trapping_b = int(trapping_b)
    b_over_a = trapping_b / trapping_a if trapping_a > 0 else np.nan
    constructed_small_b = trapping_b <= CLASSICAL_SMALL_B_THRESHOLD
    constructed_low_ratio = np.isfinite(b_over_a) and b_over_a <= CLASSICAL_B_OVER_A_THRESHOLD
    constructed_cycle_rich = feats['cycle_rank'] >= CLASSICAL_CYCLE_RICH_THRESHOLD
    trapping_like = trapping_a > 0 and (constructed_small_b or constructed_low_ratio)

    if not trapping_like:
        continue
    merged_row = dict(failure_structure_row)
    merged_row.update({
        'WL_hash': feats['WL_hash'],
        'active_detectors': int(feats['active_detectors']),
        'induced_nodes': int(feats['induced_nodes']),
        'induced_edges': int(feats['induced_edges']),
        'components': int(feats['components']),
        'largest_component_size': int(feats['largest_component_size']),
        'cycle_rank': int(feats['cycle_rank']),
        'inverse_girth': float(feats['inverse_girth']),
        'density': float(feats['density']),
        'mean_base_degree_on_active': float(feats['mean_base_degree_on_active']),
        'constructed_trapping_param': str(feats['trapping_param']),
        'constructed_trapping_a': trapping_a,
        'constructed_trapping_b': trapping_b,
        'constructed_b_over_a': float(b_over_a) if np.isfinite(b_over_a) else np.nan,
        'constructed_small_b': bool(constructed_small_b),
        'constructed_low_ratio': bool(constructed_low_ratio),
        'constructed_cycle_rich': bool(constructed_cycle_rich),
    })
    validated_adversarial_rows.append(merged_row)

assert len(validated_adversarial_rows) > 0, 'All constructed events were filtered out by trapping-set validation.'
adversarial_events_df = pd.DataFrame(validated_adversarial_rows).reset_index(drop=True)
adversarial_events_df.insert(0, 'event_id', np.arange(len(adversarial_events_df)))
adversarial_events_path = failure_structure_data_paths.output('adversarial_trapping_events_csv')
adversarial_decoder_results_long_path = failure_structure_data_paths.output('adversarial_trapping_decoder_results_long_csv')
adversarial_decoder_results_path = failure_structure_data_paths.output('adversarial_trapping_decoder_results_csv')
adversarial_decoder_summary_path = failure_structure_aggregate_path(failure_structure_data_paths.output('adversarial_trapping_decoder_summary_csv'))
adversarial_csv_dtypes_path = failure_structure_data_paths.output('adversarial_trapping_csv_dtypes_json')
adversarial_csv_dtypes = {
    adversarial_events_path.name: {
        'detector_events': 'string',
        'source_case_ids': 'string',
        'source_WL_hashes': 'string',
        'source_trapping_params': 'string',
        'WL_hash': 'string',
        'constructed_trapping_param': 'string',
    },
    adversarial_decoder_results_long_path.name: {
        'decoder': 'string',
        'status': 'string',
        'predicted_observables': 'string',
        'error_message': 'string',
    },
    adversarial_decoder_results_path.name: {
        'detector_events': 'string',
        'source_case_ids': 'string',
        'source_WL_hashes': 'string',
        'source_trapping_params': 'string',
        'WL_hash': 'string',
        'constructed_trapping_param': 'string',
        'bposd_status': 'string',
        'tesseract_status': 'string',
        'bposd_prediction': 'string',
        'tesseract_prediction': 'string',
        'bposd_error_message': 'string',
        'tesseract_error_message': 'string',
    },
    adversarial_decoder_summary_path.name: {'construction': 'string', 'constructed_trapping_param': 'string'},
}

adversarial_events_df.to_csv(adversarial_events_path, index=False, quoting=csv.QUOTE_NONNUMERIC)
decoder_result_rows = []
for (code_name, failure_structure_p), grp in tqdm(adversarial_events_df.groupby(['code', 'p']), total=adversarial_events_df.groupby(['code', 'p']).ngroups, desc='Judging BPOSD and Tesseract'):
    failure_structure_code = failure_structure_ca_codes[code_name]
    failure_structure_builder = CircuitBuilder(failure_structure_code)
    failure_structure_circ = failure_structure_builder.get_circuit_level_circuit(error_rate=ErrorRate(p=float(failure_structure_p)), d_rounds=3)
    failure_structure_dem = failure_structure_circ.detector_error_model()
    assert failure_structure_dem.num_observables > 0, f'DEM has no observables for {(code_name, failure_structure_p)}'
    decoder_factories = {
        'bposd': lambda: failure_structure_bposd(
            code=failure_structure_code,
            error_rate=float(failure_structure_p),
            max_iter=ADVERSARIAL_BPOSD_MAX_ITER,
            osd_order=ADVERSARIAL_BPOSD_OSD_ORDER,
        ).compile_decoder_for_dem(dem=failure_structure_dem),
        'tesseract': lambda: Tesseract(
            code=failure_structure_code,
            error_rate=float(failure_structure_p),
            beam=ADVERSARIAL_TESSERACT_BEAM,
            pq_limit=ADVERSARIAL_TESSERACT_PQ_LIMIT,
            seed=ADVERSARIAL_TESSERACT_SEED,
        ).compile_decoder_for_dem(dem=failure_structure_dem),
    }
    compiled_decoders = {}
    compile_errors = {}

    for decoder_name, compile_decoder in tqdm(decoder_factories.items(), desc='Trapping-set analysis', leave=False, disable=failure_structure_config['print_level'] == 1):
        try:
            compiled_decoders[decoder_name] = compile_decoder()
            compile_errors[decoder_name] = ''
        except Exception as exc:
            compiled_decoders[decoder_name] = None
            compile_errors[decoder_name] = repr(exc)

    for event in tqdm(grp.itertuples(index=False), desc='Trapping-set analysis', leave=False, disable=failure_structure_config['print_level'] == 1):
        event_vector = np.array([[1 if ch == '1' else 0 for ch in event.detector_events]], dtype=np.uint8)
        assert event_vector.shape[1] == failure_structure_dem.num_detectors, (event.event_id, event_vector.shape[1], failure_structure_dem.num_detectors)
        packed_event = np.packbits(event_vector, axis=1, bitorder='little')

        for decoder_name, failure_structure_compiled_decoder in compiled_decoders.items():
            result_row = {
                'event_id': int(event.event_id),
                'decoder': decoder_name,
                'status': 'ok',
                'predicted_observables': np.nan,
                'error_message': '',
            }

            if failure_structure_compiled_decoder is None:
                result_row['status'] = 'error'
                result_row['error_message'] = compile_errors[decoder_name]
                decoder_result_rows.append(result_row)
                continue

            try:
                pred_packed = failure_structure_compiled_decoder.decode_shots_bit_packed(bit_packed_detection_event_data=packed_event)
                pred_bits = np.unpackbits(np.asarray(pred_packed), axis=1, count=failure_structure_dem.num_observables, bitorder='little')[0]
                result_row['predicted_observables'] = ''.join(pred_bits.astype(str))
            except Exception as exc:
                result_row['status'] = 'error'
                result_row['error_message'] = repr(exc)

            decoder_result_rows.append(result_row)

adversarial_decoder_results_long = pd.DataFrame(decoder_result_rows)
adversarial_decoder_results_long.to_csv(adversarial_decoder_results_long_path, index=False, quoting=csv.QUOTE_NONNUMERIC)
status_wide = adversarial_decoder_results_long.pivot(index='event_id', columns='decoder', values='status')
status_wide = status_wide.rename(columns={c: f'{c}_status' for c in status_wide.columns}).reset_index()
pred_wide = adversarial_decoder_results_long.pivot(index='event_id', columns='decoder', values='predicted_observables')
pred_wide = pred_wide.rename(columns={c: f'{c}_prediction' for c in pred_wide.columns}).reset_index()
error_wide = adversarial_decoder_results_long.pivot(index='event_id', columns='decoder', values='error_message')
error_wide = error_wide.rename(columns={c: f'{c}_error_message' for c in error_wide.columns}).reset_index()
adversarial_decoder_results = adversarial_events_df.merge(status_wide, on='event_id', how='left').merge(pred_wide, on='event_id', how='left').merge(error_wide, on='event_id', how='left')
adversarial_decoder_results['both_decoders_ok'] = (adversarial_decoder_results['bposd_status'] == 'ok') & (adversarial_decoder_results['tesseract_status'] == 'ok')
adversarial_decoder_results['decoders_agree'] = adversarial_decoder_results['both_decoders_ok'] & (adversarial_decoder_results['bposd_prediction'] == adversarial_decoder_results['tesseract_prediction'])
adversarial_decoder_results['decoders_disagree'] = adversarial_decoder_results['both_decoders_ok'] & (adversarial_decoder_results['bposd_prediction'] != adversarial_decoder_results['tesseract_prediction'])
adversarial_decoder_results['any_decoder_error'] = ~adversarial_decoder_results['both_decoders_ok']
adversarial_decoder_results['bposd_prediction_ok'] = np.where(adversarial_decoder_results['bposd_status'] == 'ok', adversarial_decoder_results['bposd_prediction'], np.nan)
adversarial_decoder_results['tesseract_prediction_ok'] = np.where(adversarial_decoder_results['tesseract_status'] == 'ok', adversarial_decoder_results['tesseract_prediction'], np.nan)

adversarial_decoder_results.to_csv(adversarial_decoder_results_path, index=False, quoting=csv.QUOTE_NONNUMERIC)
adversarial_decoder_summary = adversarial_decoder_results.groupby(['construction', 'constructed_trapping_param']).agg(
    event_count=('event_id', 'count'),
    code_count=('code', 'nunique'),
    p_count=('p', 'nunique'),
    active_detectors_median=('active_detectors', 'median'),
    cycle_rank_median=('cycle_rank', 'median'),
    bposd_prediction_count=('bposd_prediction_ok', 'nunique'),
    tesseract_prediction_count=('tesseract_prediction_ok', 'nunique'),
    both_decoders_ok_count=('both_decoders_ok', 'sum'),
    decoder_disagreement_count=('decoders_disagree', 'sum'),
    decoder_error_count=('any_decoder_error', 'sum'),
).reset_index()
adversarial_decoder_summary['decoder_disagreement_rate_among_ok'] = np.where(adversarial_decoder_summary['both_decoders_ok_count'] > 0, adversarial_decoder_summary['decoder_disagreement_count'] / adversarial_decoder_summary['both_decoders_ok_count'], np.nan)
adversarial_decoder_summary = adversarial_decoder_summary.sort_values(['decoder_disagreement_count', 'event_count'], ascending=[False, False])

adversarial_decoder_summary.to_csv(adversarial_decoder_summary_path, index=False, quoting=csv.QUOTE_NONNUMERIC)
adversarial_csv_dtypes_path.write_text(json.dumps(adversarial_csv_dtypes, indent=2))
adversarial_events_readback = pd.read_csv(adversarial_events_path, dtype=adversarial_csv_dtypes[adversarial_events_path.name], keep_default_na=False)
assert adversarial_events_readback['detector_events'].tolist() == adversarial_events_df['detector_events'].tolist()

adversarial_decoder_results_readback = pd.read_csv(adversarial_decoder_results_path, dtype=adversarial_csv_dtypes[adversarial_decoder_results_path.name], keep_default_na=False)
assert adversarial_decoder_results_readback['detector_events'].tolist() == adversarial_decoder_results['detector_events'].tolist()
for decoder_name in tqdm(['bposd', 'tesseract'], desc='Trapping-set analysis', leave=True, disable=failure_structure_config['print_level'] == 1):
    ok_mask = adversarial_decoder_results[f'{decoder_name}_status'].to_numpy() == 'ok'
    prediction_col = f'{decoder_name}_prediction'
    assert adversarial_decoder_results_readback.loc[ok_mask, prediction_col].tolist() == adversarial_decoder_results.loc[ok_mask, prediction_col].tolist()

if failure_structure_config['print_level'] >= 2:
    print('saved:', adversarial_events_path)
if failure_structure_config['print_level'] >= 2:
    print('saved:', adversarial_decoder_results_long_path)

if failure_structure_config['print_level'] >= 2:
    print('saved:', adversarial_decoder_results_path)
if failure_structure_config['print_level'] >= 2:
    print('saved:', adversarial_decoder_summary_path)

if failure_structure_config['print_level'] >= 2:
    print('saved:', adversarial_csv_dtypes_path)
adversarial_decoder_summary.head(20)


## Trapping Set

TS on the whole tanner graph -> project the detection events onto the tanner graph -> see if there are some detection events trapped in the TS

### Test candidate detector patterns

Return to the recorded cases and evaluate the candidate structures against decoder outcomes. These tests connect a proposed motif with behavior under the selected code and priors.


In [ ]:
failure_structure_data_dir = failure_structure_data_paths.output('circuit_level')
failure_structure_cases_dir = failure_structure_data_paths.input('failure_cases')
assert failure_structure_cases_dir.is_dir(), failure_structure_cases_dir


In [ ]:
failure_structure_labels = {
    '00': 'bf',
    '11': 'bs',
    '01': 'bfts',
    '10': 'bstf',
}


In [ ]:
failure_structure_bfts_cases_path = failure_structure_data_paths.input('ca_bfts_cases_csv')
failure_structure_bstf_cases_path = failure_structure_data_paths.input('ca_bstf_cases_csv')
failure_structure_bs_cases_path = failure_structure_data_paths.input('ca_bs_cases_csv')
failure_structure_bf_cases_path = failure_structure_data_paths.input('ca_bf_cases_csv')
failure_structure_types = {
    'code': str,
    'p': float,
    'detector_events': str,
    'label': str,
}

failure_structure_bfts_cases = pd.read_csv(failure_structure_bfts_cases_path, dtype=failure_structure_types)
failure_structure_bstf_cases = pd.read_csv(failure_structure_bstf_cases_path, dtype=failure_structure_types)
failure_structure_bs_cases = pd.read_csv(failure_structure_bs_cases_path, dtype=failure_structure_types)
failure_structure_bf_cases = pd.read_csv(failure_structure_bf_cases_path, dtype=failure_structure_types)

failure_structure_bfts_cases['label'] = '01'
failure_structure_bstf_cases['label'] = '10'
failure_structure_bs_cases['label'] = '11'
failure_structure_bf_cases['label'] = '00'
failure_structure_bfts_cases.drop_duplicates(subset=['code', 'p', 'detector_events', 'label'], inplace=True)
failure_structure_bstf_cases.drop_duplicates(subset=['code', 'p', 'detector_events', 'label'], inplace=True)
failure_structure_bs_cases.drop_duplicates(subset=['code', 'p', 'detector_events', 'label'], inplace=True)
failure_structure_bf_cases.drop_duplicates(subset=['code', 'p', 'detector_events', 'label'], inplace=True)


In [ ]:
analysis_rows = []
for (code_name, failure_structure_p), grp_fail in tqdm(failure_structure_bfts_cases.groupby(['code', 'p']), desc='Trapping-set analysis', leave=True, disable=failure_structure_config['print_level'] == 1):
    if failure_structure_config['print_level'] >= 2:
        print(f'Analyzing failure cases for code={code_name}, p={failure_structure_p:.6f} with {len(grp_fail)} samples')
    failure_structure_code = failure_structure_ca_codes[code_name]
    failure_structure_builder = CircuitBuilder(failure_structure_code)
    failure_structure_circ = failure_structure_builder.get_circuit(noise_model='circuit_level', error_rate=ErrorRate(p=failure_structure_p), d_rounds=3)
    failure_structure_dem = failure_structure_circ.detector_error_model()
    failure_structure_g, failure_structure_ = failure_structure_build_tanner_graph_from_dem(failure_structure_dem)
    ctrl_df = failure_structure_bs_cases[(failure_structure_bs_cases['code'] == code_name) & (failure_structure_bs_cases['p'] == failure_structure_p)][['code', 'p', 'detector_events', 'label']].copy()
    fail_df = grp_fail[['code', 'p', 'detector_events', 'label']].copy()
    merged = pd.concat([fail_df, ctrl_df], ignore_index=True)

    for failure_structure_, failure_structure_r in tqdm(merged.iterrows(), desc='Trapping-set analysis', leave=False, disable=failure_structure_config['print_level'] == 1):
        feats = graph_features_no_det_event(failure_structure_g)
        feats.update({
            'code': code_name,
            'p': float(failure_structure_p),
            'label': failure_structure_r['label'],
            'detector_events': failure_structure_r['detector_events'],
        })
        analysis_rows.append(feats)

analysis_df = pd.DataFrame(analysis_rows)
analysis_df.head()


In [ ]:
failure_structure_code = CA_5_2_3()
failure_structure_builder = CircuitBuilder(failure_structure_code)
failure_structure_circ = failure_structure_builder.get_circuit(noise_model='circuit_level', error_rate=ErrorRate(p=0.001111), d_rounds=3)
failure_structure_dem = failure_structure_circ.detector_error_model()
failure_structure_g, failure_structure_ = failure_structure_build_tanner_graph_from_dem(failure_structure_dem)


In [ ]:
save_plotly(visualize_det_on_graph_plotly(failure_structure_g, 'tanner', '000011110000'), failure_structure_data_paths.output('cell_0072_1_html')).show()
